# Submission: XGBoost

Standalone pipeline: load data → build features → train XGBoost on all data → predict test → validate & submit.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import xgboost as xgb

from src import (
    load_train_speeds, load_train_texts, load_test_data,
    load_adjacency, load_roads,
    build_windows, build_features,
    write_submission, validate_submission,
)

In [ ]:
s1, s2 = load_train_speeds()
t1, t2 = load_train_texts()
adj = load_adjacency()
roads = load_roads()

X1, T1, Y1 = build_windows(s1, t1)
X2, T2, Y2 = build_windows(s2, t2)

X_all = np.concatenate([X1, X2], axis=0).astype(np.float32)
Y_all = np.concatenate([Y1, Y2], axis=0).astype(np.float32)

print(f"Train samples: {len(X_all)}")

In [ ]:
print("Building features...")
F_all = build_features(X_all, adj, roads)
Y_flat = Y_all.transpose(0, 2, 1).reshape(-1, 3)
print(f"Features: {F_all.shape}, Targets: {Y_flat.shape}")

In [ ]:
models = {}
for hi, hname in enumerate(["h5", "h10", "h15"]):
    model = xgb.XGBRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, verbosity=0, n_jobs=-1,
    )
    model.fit(F_all, Y_flat[:, hi])
    models[hname] = model
    print(f"  {hname} done")

In [ ]:
test_hist, _ = load_test_data()
F_test = build_features(test_hist, adj, roads)
print(f"Test features: {F_test.shape}")

test_preds = np.zeros((540, 3, 1260), dtype=np.float32)
for hi, hname in enumerate(["h5", "h10", "h15"]):
    test_preds[:, hi, :] = models[hname].predict(F_test).reshape(540, 1260)
test_preds = test_preds.clip(min=0)

print(f"Predictions: {test_preds.shape}, range=[{test_preds.min():.1f}, {test_preds.max():.1f}]")

In [ ]:
out_dir = write_submission(test_preds, label="xgb_final", models=models)
validate_submission(out_dir / "submission.csv")